# Day2+ Geometry-Aware Transferability Analysis

This notebook extends the Day2+ cross-board transfer stress test by asking a more mechanistic question:

> Can source-target geometry predict calibration transfer difficulty?

The working hypothesis is that transfer error is not controlled only by model choice. Instead, it should increase when the target board occupies a feature-space manifold that is far from, or dynamically shaped differently from, the source manifold.

Primary outputs are saved under:

- `figures/day2plus/`
- `results/day2plus/`

In [1]:
from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.metrics import r2_score

# Robust project-root detection for running either from notebooks/ or project root.
cwd = Path.cwd().resolve()
if (cwd / "src").exists() and (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists() and (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    # Fallback for copied notebooks: search upward.
    candidates = [cwd, *cwd.parents]
    PROJECT_ROOT = next((p for p in candidates if (p / "src").exists() and (p / "data").exists()), cwd.parent)

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIG_DIR = PROJECT_ROOT / "figures" / "day2plus"
RESULT_DIR = PROJECT_ROOT / "results" / "day2plus"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("SRC_DIR:", SRC_DIR)
print("FIG_DIR:", FIG_DIR)
print("RESULT_DIR:", RESULT_DIR)

PROJECT_ROOT: C:\Users\hg\PycharmProjects\mox_calibration_transfer
DATA_DIR: C:\Users\hg\PycharmProjects\mox_calibration_transfer\data\raw
SRC_DIR: C:\Users\hg\PycharmProjects\mox_calibration_transfer\src
FIG_DIR: C:\Users\hg\PycharmProjects\mox_calibration_transfer\figures\day2plus
RESULT_DIR: C:\Users\hg\PycharmProjects\mox_calibration_transfer\results\day2plus


In [2]:
from day2plus_transfer_matrix import (
    HAS_XGBOOST,
    TARGET_GAS,
    assert_no_label_leakage,
    build_feature_table,
    get_feature_columns,
    run_single_source_matrix,
    run_mixed_domain_combinations,
)

FEATURE_SET = "physics"          # Primary Day2+ representation
MODEL_NAME = "RandomForest"      # Primary lightweight nonlinear model
RANDOM_STATE = 42

MODEL_NAMES = ["LinearRegression", "RandomForest"]
if HAS_XGBOOST:
    MODEL_NAMES.append("XGBoost")

print("Models available:", MODEL_NAMES)

Models available: ['LinearRegression', 'RandomForest', 'XGBoost']


## 1. Load or build the methane feature table

The geometry metrics must be computed only from input features. Concentration labels are used only for transfer-error evaluation, not for geometry calculation.

In [3]:
feature_cache = RESULT_DIR / "day2plus_feature_table_GMe.csv"

if feature_cache.exists():
    df = pd.read_csv(feature_cache)
    print("Loaded cached feature table:", feature_cache)
else:
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Missing raw data directory: {DATA_DIR}")
    df = build_feature_table(DATA_DIR, gas=TARGET_GAS)
    df.to_csv(feature_cache, index=False)
    print("Built and cached feature table:", feature_cache)

print("Shape:", df.shape)
print("Boards:", sorted(df["board"].unique()))
print("Concentrations:", sorted(df["concentration_numeric"].unique()))

leakage_report = assert_no_label_leakage(df)
leakage_report

Built and cached feature table: C:\Users\hg\PycharmProjects\mox_calibration_transfer\results\day2plus\day2plus_feature_table_GMe.csv
Shape: (160, 126)
Boards: ['B1', 'B2', 'B3', 'B4', 'B5']
Concentrations: [np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(60), np.int64(70), np.int64(80), np.int64(90), np.int64(100)]


{'raw': {'n_features': 48, 'leakage_overlap': [], 'ok': True},
 'physics': {'n_features': 71, 'leakage_overlap': [], 'ok': True},
 'combined': {'n_features': 119, 'leakage_overlap': [], 'ok': True}}

## 2. Load or compute transfer metrics

This notebook correlates geometry metrics with actual transfer error. It first tries to load existing Day2+ result CSV files. If they are not present, it recomputes the required RandomForest physics transfer experiments.

In [4]:
def find_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

single_candidates = [
    RESULT_DIR / "single_source_metrics.csv",
    RESULT_DIR / "day2plus_single_source_metrics.csv",
    RESULT_DIR / "single_metrics.csv",
]
mixed_candidates = [
    RESULT_DIR / "mixed_domain_metrics.csv",
    RESULT_DIR / "day2plus_mixed_domain_metrics.csv",
    RESULT_DIR / "mixed_metrics.csv",
]

single_path = find_first_existing(single_candidates)
mixed_path = find_first_existing(mixed_candidates)

if single_path is not None:
    single_metrics = pd.read_csv(single_path)
    print("Loaded single-source metrics:", single_path)
else:
    print("No single-source metrics found. Recomputing physics/RandomForest single-source matrix...")
    single_metrics, _single_preds = run_single_source_matrix(
        df,
        feature_sets=[FEATURE_SET],
        model_names=[MODEL_NAME],
        random_state=RANDOM_STATE,
    )
    single_metrics.to_csv(RESULT_DIR / "geometry_single_source_metrics.csv", index=False)

if mixed_path is not None:
    mixed_metrics = pd.read_csv(mixed_path)
    print("Loaded mixed-domain metrics:", mixed_path)
else:
    print("No mixed-domain metrics found. Recomputing physics/RandomForest mixed-domain combinations...")
    mixed_metrics, _mixed_preds = run_mixed_domain_combinations(
        df,
        source_sizes=(2, 3, 4),
        feature_set=FEATURE_SET,
        model_name=MODEL_NAME,
        random_state=RANDOM_STATE,
    )
    mixed_metrics.to_csv(RESULT_DIR / "geometry_mixed_domain_metrics.csv", index=False)

metrics_df = pd.concat([single_metrics, mixed_metrics], ignore_index=True)
metrics_df = metrics_df[(metrics_df["feature_set"] == FEATURE_SET) & (metrics_df["model"] == MODEL_NAME)].copy()
metrics_df = metrics_df.drop_duplicates(subset=["source_boards", "target_board", "feature_set", "model"])

print("Geometry/evaluation pairs:", len(metrics_df))
metrics_df.head()

No single-source metrics found. Recomputing physics/RandomForest single-source matrix...
No mixed-domain metrics found. Recomputing physics/RandomForest mixed-domain combinations...
Geometry/evaluation pairs: 75


,source_boards,n_source_boards,target_board,feature_set,model,n_train,n_test,rmse,mae,r2,includes_B5_source
0,B1,1,B2,physics,RandomForest,40,40,6.237779,4.825625,0.952836,False
1,B1,1,B3,physics,RandomForest,40,40,10.385157,9.590625,0.869271,False
2,B1,1,B4,physics,RandomForest,40,20,9.155391,8.202500,0.898399,False
3,B1,1,B5,physics,RandomForest,40,20,7.793647,7.287500,0.926375,False
4,B2,1,B1,physics,RandomForest,40,40,8.160622,6.738750,0.919278,False


## 3. Compute source-target geometry metrics

For each transfer split, we calculate geometry mismatch in the same feature representation used by the model.

Metrics:

- `centroid_dist`: Euclidean distance between source and target feature centroids.
- `cov_fro_dist`: Frobenius distance between source and target covariance matrices.
- `cov_fro_dist_norm`: covariance distance normalized by source and target covariance magnitudes.
- `pca_centroid_dist`: centroid distance after projecting all samples into a common PCA space.
- `pca_cov_fro_dist`: covariance distance in PCA space.

Interpretation:

- Centroid mismatch approximates baseline/domain shift.
- Covariance mismatch approximates dynamic deformation, spread, and kinetic instability.

In [5]:
def parse_source_boards(source_boards):
    if isinstance(source_boards, (list, tuple, set)):
        return sorted([str(b).upper() for b in source_boards])
    return sorted([b.strip().upper() for b in str(source_boards).split("+") if b.strip()])


def safe_cov(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    if X.shape[0] < 2:
        return np.zeros((X.shape[1], X.shape[1]))
    return np.cov(X, rowvar=False)


def geometry_for_split(df, source_boards, target_board, feature_set="physics", pca_components=5):
    source_boards = parse_source_boards(source_boards)
    target_board = str(target_board).upper()
    if target_board in source_boards:
        raise ValueError("target_board must not be in source_boards")

    cols = get_feature_columns(df, feature_set)
    all_part = df[df["board"].isin(source_boards + [target_board])].copy()
    source_part = all_part[all_part["board"].isin(source_boards)].copy()
    target_part = all_part[all_part["board"].eq(target_board)].copy()

    if source_part.empty or target_part.empty:
        raise ValueError(f"Empty split: source={source_boards}, target={target_board}")

    X_all = all_part[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    X_source_raw = source_part[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    X_target_raw = target_part[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)

    scaler = StandardScaler().fit(X_all)
    Xs = scaler.transform(X_source_raw)
    Xt = scaler.transform(X_target_raw)

    mu_s = Xs.mean(axis=0)
    mu_t = Xt.mean(axis=0)
    cov_s = safe_cov(Xs)
    cov_t = safe_cov(Xt)

    centroid_dist = float(np.linalg.norm(mu_s - mu_t))
    cov_fro_dist = float(np.linalg.norm(cov_s - cov_t, ord="fro"))
    cov_norm_den = float(np.linalg.norm(cov_s, ord="fro") + np.linalg.norm(cov_t, ord="fro") + 1e-12)
    cov_fro_dist_norm = float(cov_fro_dist / cov_norm_den)

    n_components = int(min(pca_components, X_all.shape[0] - 1, X_all.shape[1]))
    if n_components >= 1:
        X_all_scaled = scaler.transform(X_all)
        pca = PCA(n_components=n_components, random_state=RANDOM_STATE).fit(X_all_scaled)
        Ps = pca.transform(Xs)
        Pt = pca.transform(Xt)
        pca_mu_s = Ps.mean(axis=0)
        pca_mu_t = Pt.mean(axis=0)
        pca_cov_s = safe_cov(Ps)
        pca_cov_t = safe_cov(Pt)
        pca_centroid_dist = float(np.linalg.norm(pca_mu_s - pca_mu_t))
        pca_cov_fro_dist = float(np.linalg.norm(pca_cov_s - pca_cov_t, ord="fro"))
        pca_explained = float(pca.explained_variance_ratio_.sum())
    else:
        pca_centroid_dist = np.nan
        pca_cov_fro_dist = np.nan
        pca_explained = np.nan

    return {
        "source_boards": "+".join(source_boards),
        "n_source_boards": len(source_boards),
        "target_board": target_board,
        "feature_set": feature_set,
        "n_source_samples": len(source_part),
        "n_target_samples": len(target_part),
        "centroid_dist": centroid_dist,
        "cov_fro_dist": cov_fro_dist,
        "cov_fro_dist_norm": cov_fro_dist_norm,
        "pca_centroid_dist": pca_centroid_dist,
        "pca_cov_fro_dist": pca_cov_fro_dist,
        "pca_explained_var": pca_explained,
    }

geometry_rows = []
for _, row in metrics_df.iterrows():
    geometry_rows.append(
        geometry_for_split(
            df,
            row["source_boards"],
            row["target_board"],
            feature_set=row["feature_set"],
            pca_components=5,
        )
    )

geometry_df = pd.DataFrame(geometry_rows)
geometry_transfer = metrics_df.merge(
    geometry_df,
    on=["source_boards", "n_source_boards", "target_board", "feature_set"],
    how="left",
)

out_path = RESULT_DIR / "geometry_transfer_metrics.csv"
geometry_transfer.to_csv(out_path, index=False)
print("Saved:", out_path)
geometry_transfer.head()

Saved: C:\Users\hg\PycharmProjects\mox_calibration_transfer\results\day2plus\geometry_transfer_metrics.csv


,source_boards,n_source_boards,target_board,feature_set,model,n_train,n_test,rmse,mae,r2,includes_B5_source,n_source_samples,n_target_samples,centroid_dist,cov_fro_dist,cov_fro_dist_norm,pca_centroid_dist,pca_cov_fro_dist,pca_explained_var
0,B1,1,B2,physics,RandomForest,40,40,6.237779,4.825625,0.952836,False,40,40,6.316743,18.040970,0.266111,6.304347,13.308190,0.873165
1,B1,1,B3,physics,RandomForest,40,40,10.385157,9.590625,0.869271,False,40,40,6.615387,19.146598,0.300471,6.610855,13.808458,0.850988
2,B1,1,B4,physics,RandomForest,40,20,9.155391,8.202500,0.898399,False,40,20,7.788446,18.839343,0.292122,7.776993,14.507654,0.876554
3,B1,1,B5,physics,RandomForest,40,20,7.793647,7.287500,0.926375,False,40,20,7.524697,56.215643,0.652587,7.491721,54.461235,0.915812
4,B2,1,B1,physics,RandomForest,40,40,8.160622,6.738750,0.919278,False,40,40,6.316743,18.040970,0.266111,6.304347,13.308190,0.873165


## 4. Geometry-transfer correlation

If geometry predicts transferability, RMSE should increase with one or more source-target geometry distances.

In [6]:
def corr_table(df, y="rmse", xs=None):
    if xs is None:
        xs = [
            "centroid_dist",
            "cov_fro_dist",
            "cov_fro_dist_norm",
            "pca_centroid_dist",
            "pca_cov_fro_dist",
        ]
    rows = []
    tmp = df[[y, *xs]].replace([np.inf, -np.inf], np.nan).dropna()
    for x in xs:
        pearson = tmp[[x, y]].corr(method="pearson").iloc[0, 1]
        spearman = tmp[[x, y]].corr(method="spearman").iloc[0, 1]
        rows.append({"metric": x, "pearson_r": pearson, "spearman_r": spearman, "n": len(tmp)})
    return pd.DataFrame(rows).sort_values("pearson_r", key=lambda s: s.abs(), ascending=False)

corr_all = corr_table(geometry_transfer)
corr_single = corr_table(geometry_transfer[geometry_transfer["n_source_boards"] == 1])
corr_mixed = corr_table(geometry_transfer[geometry_transfer["n_source_boards"] > 1])

corr_all.to_csv(RESULT_DIR / "geometry_correlation_all.csv", index=False)
corr_single.to_csv(RESULT_DIR / "geometry_correlation_single_source.csv", index=False)
corr_mixed.to_csv(RESULT_DIR / "geometry_correlation_mixed_source.csv", index=False)

print("All splits")
display(corr_all)
print("Single-source only")
display(corr_single)
print("Mixed-source only")
display(corr_mixed)

All splits


,metric,pearson_r,spearman_r,n
4,pca_cov_fro_dist,-0.419852,-0.458653,75
1,cov_fro_dist,-0.414465,-0.440842,75
2,cov_fro_dist_norm,-0.385784,-0.450601,75
3,pca_centroid_dist,0.361436,0.287830,75
0,centroid_dist,0.292420,0.150893,75


Single-source only


,metric,pearson_r,spearman_r,n
0,centroid_dist,0.553576,0.582646,20
3,pca_centroid_dist,0.547471,0.582646,20
4,pca_cov_fro_dist,-0.269941,-0.250568,20
1,cov_fro_dist,-0.269614,-0.250568,20
2,cov_fro_dist_norm,-0.250504,-0.156982,20


Mixed-source only


,metric,pearson_r,spearman_r,n
4,pca_cov_fro_dist,-0.472071,-0.495022,55
1,cov_fro_dist,-0.466389,-0.478427,55
2,cov_fro_dist_norm,-0.435962,-0.485498,55
3,pca_centroid_dist,0.210379,0.065079,55
0,centroid_dist,0.073033,-0.083694,55


### Interpretation checkpoint

Use the table above to decide whether transfer difficulty is more strongly associated with mean-domain shift (`centroid_dist`) or dynamic manifold deformation (`cov_fro_dist` / `pca_cov_fro_dist`).

A stronger covariance-distance correlation would support the interpretation that board transfer failure is driven more by dynamic mismatch than by simple baseline offset.

## 5. RMSE versus geometry distances

In [7]:
def scatter_with_fit(df, x, y, path, title=None, annotate=True):
    tmp = df[[x, y, "source_boards", "target_board", "n_source_boards"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    fig, ax = plt.subplots(figsize=(6.8, 5.0), dpi=160)
    sc = ax.scatter(tmp[x], tmp[y], c=tmp["n_source_boards"], s=55, alpha=0.85)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Number of source boards")

    if len(tmp) >= 2 and tmp[x].nunique() > 1:
        coef = np.polyfit(tmp[x], tmp[y], deg=1)
        xx = np.linspace(tmp[x].min(), tmp[x].max(), 100)
        yy = coef[0] * xx + coef[1]
        ax.plot(xx, yy, linestyle="--", linewidth=1.5)
        pearson = tmp[[x, y]].corr(method="pearson").iloc[0, 1]
        spearman = tmp[[x, y]].corr(method="spearman").iloc[0, 1]
        label = f"Pearson r={pearson:.2f}\nSpearman ρ={spearman:.2f}"
        ax.text(0.02, 0.98, label, transform=ax.transAxes, va="top", ha="left",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    if annotate:
        # Annotate the highest-error cases, which are scientifically most informative.
        for _, row in tmp.sort_values(y, ascending=False).head(6).iterrows():
            ax.annotate(f"{row['source_boards']}→{row['target_board']}",
                        (row[x], row[y]), fontsize=7, xytext=(4, 4), textcoords="offset points")

    ax.set_xlabel(x)
    ax.set_ylabel(y.upper())
    ax.set_title(title or f"{y.upper()} vs {x}")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return tmp

scatter_with_fit(
    geometry_transfer,
    "centroid_dist",
    "rmse",
    FIG_DIR / "geometry_rmse_vs_centroid_distance.png",
    title="Transfer RMSE vs centroid mismatch",
)
scatter_with_fit(
    geometry_transfer,
    "cov_fro_dist",
    "rmse",
    FIG_DIR / "geometry_rmse_vs_covariance_distance.png",
    title="Transfer RMSE vs covariance mismatch",
)
scatter_with_fit(
    geometry_transfer,
    "pca_cov_fro_dist",
    "rmse",
    FIG_DIR / "geometry_rmse_vs_pca_covariance_distance.png",
    title="Transfer RMSE vs PCA covariance mismatch",
)

print("Saved scatter figures to:", FIG_DIR)

Saved scatter figures to: C:\Users\hg\PycharmProjects\mox_calibration_transfer\figures\day2plus


## 6. Two-dimensional geometry map

This figure asks whether high-transfer-error splits cluster in a geometry space defined by centroid mismatch and covariance mismatch.

In [8]:
def plot_geometry_map(df, path):
    tmp = df[["centroid_dist", "cov_fro_dist", "rmse", "source_boards", "target_board", "n_source_boards", "includes_B5_source"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    fig, ax = plt.subplots(figsize=(7.2, 5.4), dpi=160)
    marker_map = {False: "o", True: "X"}
    for includes_b5, part in tmp.groupby("includes_B5_source"):
        sc = ax.scatter(
            part["centroid_dist"],
            part["cov_fro_dist"],
            c=part["rmse"],
            s=45 + 25 * part["n_source_boards"],
            marker=marker_map.get(bool(includes_b5), "o"),
            alpha=0.85,
            label=f"B5 source={includes_b5}",
        )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("RMSE")
    for _, row in tmp.sort_values("rmse", ascending=False).head(8).iterrows():
        ax.annotate(f"{row['source_boards']}→{row['target_board']}",
                    (row["centroid_dist"], row["cov_fro_dist"]), fontsize=7,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("Centroid distance")
    ax.set_ylabel("Covariance Frobenius distance")
    ax.set_title("Geometry map of transfer difficulty")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

plot_geometry_map(geometry_transfer, FIG_DIR / "geometry_map_centroid_covariance_rmse.png")
print("Saved:", FIG_DIR / "geometry_map_centroid_covariance_rmse.png")

Saved: C:\Users\hg\PycharmProjects\mox_calibration_transfer\figures\day2plus\geometry_map_centroid_covariance_rmse.png


## 7. Simple predictive check: can geometry predict RMSE?

This is not meant as a final predictive model. It is a sanity check: if a simple linear model using only geometry metrics explains meaningful RMSE variance, then geometry is carrying transferability information.

In [9]:
def fit_geometry_predictor(df, feature_cols, y_col="rmse", label="all"):
    tmp = df[[*feature_cols, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(tmp) < 5:
        return {"label": label, "n": len(tmp), "r2_in_sample": np.nan}
    X = tmp[feature_cols].to_numpy(dtype=float)
    y = tmp[y_col].to_numpy(dtype=float)
    model = SkLinearRegression().fit(X, y)
    pred = model.predict(X)
    return {
        "label": label,
        "n": len(tmp),
        "r2_in_sample": float(r2_score(y, pred)),
        **{f"coef_{c}": float(v) for c, v in zip(feature_cols, model.coef_)},
        "intercept": float(model.intercept_),
    }

geom_features = ["centroid_dist", "cov_fro_dist", "cov_fro_dist_norm", "pca_centroid_dist", "pca_cov_fro_dist"]
models = [
    fit_geometry_predictor(geometry_transfer, ["centroid_dist"], label="centroid_only"),
    fit_geometry_predictor(geometry_transfer, ["cov_fro_dist"], label="covariance_only"),
    fit_geometry_predictor(geometry_transfer, ["centroid_dist", "cov_fro_dist"], label="centroid_plus_covariance"),
    fit_geometry_predictor(geometry_transfer, geom_features, label="all_geometry_metrics"),
]
predictor_df = pd.DataFrame(models)
predictor_df.to_csv(RESULT_DIR / "geometry_predict_rmse_linear_check.csv", index=False)
predictor_df

,label,n,r2_in_sample,coef_centroid_dist,intercept,coef_cov_fro_dist,coef_cov_fro_dist_norm,coef_pca_centroid_dist,coef_pca_cov_fro_dist
0,centroid_only,75,0.085509,0.790512,2.523458,NaN,NaN,NaN,NaN
1,covariance_only,75,0.171781,NaN,8.699944,-0.048567,NaN,NaN,NaN
2,centroid_plus_covariance,75,0.424324,1.484940,1.322026,-0.074555,NaN,NaN,NaN
3,all_geometry_metrics,75,0.463263,0.557392,0.235603,0.007836,8.860914,0.733165,-0.135183


## 8. Board-level summaries

These summaries identify which targets are geometrically isolated and which source combinations create large manifold mismatch.

In [10]:
target_summary = geometry_transfer.groupby("target_board", as_index=False).agg(
    mean_rmse=("rmse", "mean"),
    mean_centroid_dist=("centroid_dist", "mean"),
    mean_cov_fro_dist=("cov_fro_dist", "mean"),
    mean_pca_cov_fro_dist=("pca_cov_fro_dist", "mean"),
    n_splits=("rmse", "size"),
).sort_values("mean_rmse", ascending=False)

source_summary = geometry_transfer.groupby("source_boards", as_index=False).agg(
    mean_rmse=("rmse", "mean"),
    mean_centroid_dist=("centroid_dist", "mean"),
    mean_cov_fro_dist=("cov_fro_dist", "mean"),
    n_source_boards=("n_source_boards", "first"),
    includes_B5_source=("includes_B5_source", "first"),
    n_targets=("target_board", "nunique"),
).sort_values("mean_rmse", ascending=False)

target_summary.to_csv(RESULT_DIR / "geometry_target_difficulty_summary.csv", index=False)
source_summary.to_csv(RESULT_DIR / "geometry_source_robustness_summary.csv", index=False)

print("Target difficulty summary")
display(target_summary)
print("Source robustness summary")
display(source_summary.head(15))

Target difficulty summary


,target_board,mean_rmse,mean_centroid_dist,mean_cov_fro_dist,mean_pca_cov_fro_dist,n_splits
0,B1,11.105836,6.263678,25.378236,21.833852,15
1,B2,6.859121,4.552486,25.740598,22.778313,15
2,B3,6.527000,4.871665,25.623792,21.562732,15
3,B4,5.612242,5.751514,23.921814,21.687670,15
4,B5,4.644222,6.556598,79.524381,75.717229,15


Source robustness summary


,source_boards,mean_rmse,mean_centroid_dist,mean_cov_fro_dist,n_source_boards,includes_B5_source,n_targets
27,B4,11.675417,6.804073,26.174133,1,False,4
25,B3+B4+B5,11.628443,4.782420,26.299359,3,True,2
18,B2+B3+B4+B5,9.752295,5.671207,22.791209,4,True,1
26,B3+B5,8.554853,4.935077,28.786656,2,True,3
0,B1,8.392994,7.061318,28.060639,1,False,4
24,B3+B4,8.113230,5.779302,34.998772,2,False,3
29,B5,7.944368,6.452264,53.026785,1,True,4
17,B2+B3+B4,7.463924,6.295523,57.260423,3,False,2
28,B4+B5,7.215252,4.980488,37.389968,2,True,3
19,B2+B3+B5,7.131795,5.337684,23.649276,3,True,2


## 9. Scientific interpretation template

After running this notebook, fill in the observed correlation values from Section 4.

Suggested interpretation patterns:

1. If `cov_fro_dist` correlates more strongly with RMSE than `centroid_dist`:
   - Transfer failure is better explained by dynamic manifold deformation than by simple baseline shift.
   - This supports the Day2+ observation that B5-like behavior is not merely offset drift.

2. If `centroid_dist` is stronger:
   - Baseline/domain displacement remains the dominant transfer barrier.
   - Physics-informed normalization helps but does not fully align absolute calibration across boards.

3. If both are weak:
   - Transfer difficulty may be concentration-regime-specific rather than globally geometric.
   - In that case, repeat geometry calculation by concentration band: low, medium, high.

Key sentence for the Day2+ report:

> Transferability is not determined by source diversity alone. Geometry-aware analysis separates useful manifold coverage from destructive heterogeneity by quantifying source-target centroid and covariance mismatch.